In [ ]:
import os
import json
import pyarrow.parquet as pq
from mindspore.mindrecord import FileWriter

# =========================
# 路径配置
# =========================
PARQUET_FILE = r"multi-chart-infographic-reasoning/test.parquet"

# 数据集根目录
DATA_ROOT = r"C:multi-chart-infographic-reasoning"

# 输出 MindRecord
MINDRECORD_FILE = r"C:multi-chart-infographic-reasoning/test.mindrecord"

# =========================
# 读取 parquet
# =========================
print("Loading parquet...")

table = pq.read_table(PARQUET_FILE)
samples = table.to_pylist()

print(f"Total samples: {len(samples)}")

# =========================
# 创建 MindRecord Writer
# =========================
writer = FileWriter(
    file_name=MINDRECORD_FILE,
    shard_num=1
)

schema = {
    "image": {"type": "bytes"},
    "image_path": {"type": "string"},
    "chart_types": {"type": "string"},
    "category": {"type": "string"},
    "source": {"type": "string"},
    "questions": {"type": "string"},
    "answers": {"type": "string"},
    "question_types": {"type": "string"}
}

writer.add_schema(schema, "multi_chart_infographic")

# =========================
# 构建数据
# =========================
records = []

for idx, sample in enumerate(samples):

    image_rel_path = sample.get("image_path", "")
    image_abs_path = os.path.join(DATA_ROOT, image_rel_path)

    # 图片不存在则跳过
    if not os.path.exists(image_abs_path):
        print(f"[Skip] Image not found: {image_abs_path}")
        continue

    try:
        with open(image_abs_path, "rb") as f:
            image_bytes = f.read()
    except Exception as e:
        print(f"[Skip] Failed to read image: {image_abs_path}")
        print(e)
        continue

    questions = []
    answers = []
    question_types = []

    for i in range(1, 7):

        q = sample.get(f"sub_question {i}", "")
        a = sample.get(f"answer {i}", "")
        t = sample.get(f"question_type {i}", "")

        if q and str(q).strip():
            questions.append(q)

        if a and str(a).strip():
            answers.append(a)

        if t and str(t).strip():
            question_types.append(t)

    record = {
        "image": image_bytes,
        "image_path": image_rel_path,
        "chart_types": json.dumps(
            sample.get("chart_types", []),
            ensure_ascii=False
        ),
        "category": sample.get("category", ""),
        "source": sample.get("source", ""),
        "questions": json.dumps(
            questions,
            ensure_ascii=False
        ),
        "answers": json.dumps(
            answers,
            ensure_ascii=False
        ),
        "question_types": json.dumps(
            question_types,
            ensure_ascii=False
        )
    }

    records.append(record)

    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1}/{len(samples)}")

# =========================
# 写入 MindRecord
# =========================
print(f"Writing {len(records)} samples...")

writer.write_raw_data(records)
writer.commit()

print("Done!")
print(f"Saved to: {MINDRECORD_FILE}")

In [ ]:
import json
import mindspore.dataset as ds

dataset = ds.MindDataset(
    dataset_files="multi-chart-infographic-reasoning/test.mindrecord",
    shuffle=False
)

for item in dataset.create_dict_iterator():

    image = item["image"]          # bytes
    image_path = item["image_path"]

    questions = json.loads(item["questions"].item())
    answers = json.loads(item["answers"].item())
    question_types = json.loads(item["question_types"].item())

    print(image_path)
    print(questions)
    print(answers)

    break